# Notebook 21 — Experiment Tracking and Reporting

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/evals/21_experiment_tracking_and_reporting.ipynb)

By the time you have multiple prompt versions, safety runs, and human-review passes, raw metrics are not enough. In this notebook we build local reporting utilities for Castalia Bench: run metadata, leaderboards, reproducibility logs, and comparison reports.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **What you will build**
- Understand **Reporting principle**
- Understand **Step 1: Define the experiment records**
- Understand **Step 2: Validate and normalize the schema**
- Understand **Step 3: Compute a composite score**


In [ ]:
# --- Google Colab Setup ---
!pip install -q "transformers>=4.51.0" accelerate bitsandbytes torch

import csv
import json
import math
import random
import statistics
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True, top_k=20):
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=top_k,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## What you will build

- a run schema with reproducibility metadata
- leaderboard tables for multiple experiment families
- comparison utilities versus a baseline release
- reproducibility logs and config hashes
- reporting artifacts that can feed release reviews

## Reporting principle

Experiment tracking is not about collecting as many fields as possible. It is about collecting the fields needed to answer:

- what changed
- what improved
- what regressed
- can this run be reproduced
- should this variant be promoted

In [ ]:
from collections import defaultdict
from datetime import datetime
from pathlib import Path
import hashlib
import json
import random
import statistics

random.seed(21)

ARTIFACT_DIR = Path("artifacts") / "notebook_21_reporting"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

def to_markdown_table(rows, columns):
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(column, "")) for column in columns) + " |")
    return "\n".join([header, divider, *body])

print("Artifact directory:", ARTIFACT_DIR.resolve())

## Step 1: Define the experiment records

Each run captures:

- system identity
- dataset and benchmark version
- quality, safety, latency, and cost metrics
- a release-gate outcome
- reproducibility fields like seed, git sha, and config details

In [ ]:
run_records = [
    {
        "run_id": "prompt_v1",
        "family": "prompt",
        "system_name": "Prompt Helper",
        "variant": "baseline",
        "dataset_version": "castalia_bench_v0.4",
        "quality_score": 0.74,
        "safety_score": 0.91,
        "human_score": 0.71,
        "latency_ms": 880,
        "cost_usd": 0.012,
        "release_gate": "ship",
        "git_sha": "a1b2c3d",
        "seed": 21,
        "prompt_version": "support_prompt@1",
        "eval_config": {"judge": "none", "regression_suite": "m5-smoke"},
        "case_scores": [0.68, 0.75, 0.72, 0.8, 0.74],
    },
    {
        "run_id": "prompt_v2",
        "family": "prompt",
        "system_name": "Prompt Helper",
        "variant": "candidate",
        "dataset_version": "castalia_bench_v0.4",
        "quality_score": 0.81,
        "safety_score": 0.93,
        "human_score": 0.78,
        "latency_ms": 920,
        "cost_usd": 0.014,
        "release_gate": "ship",
        "git_sha": "d4e5f6g",
        "seed": 21,
        "prompt_version": "support_prompt@2",
        "eval_config": {"judge": "none", "regression_suite": "m5-smoke"},
        "case_scores": [0.79, 0.83, 0.76, 0.86, 0.81],
    },
    {
        "run_id": "rag_v1",
        "family": "rag",
        "system_name": "RAG Assistant",
        "variant": "baseline",
        "dataset_version": "castalia_bench_v0.4",
        "quality_score": 0.78,
        "safety_score": 0.89,
        "human_score": 0.76,
        "latency_ms": 1450,
        "cost_usd": 0.026,
        "release_gate": "review",
        "git_sha": "r1g2h3i",
        "seed": 21,
        "prompt_version": "rag_prompt@1",
        "eval_config": {"judge": "rubric_qwen", "retrieval_k": 5},
        "case_scores": [0.73, 0.82, 0.77, 0.8, 0.78],
    },
    {
        "run_id": "rag_v2",
        "family": "rag",
        "system_name": "RAG Assistant",
        "variant": "candidate",
        "dataset_version": "castalia_bench_v0.4",
        "quality_score": 0.84,
        "safety_score": 0.92,
        "human_score": 0.79,
        "latency_ms": 1310,
        "cost_usd": 0.024,
        "release_gate": "ship",
        "git_sha": "j4k5l6m",
        "seed": 21,
        "prompt_version": "rag_prompt@2",
        "eval_config": {"judge": "rubric_qwen", "retrieval_k": 8},
        "case_scores": [0.82, 0.85, 0.8, 0.88, 0.84],
    },
    {
        "run_id": "agent_v1",
        "family": "agent",
        "system_name": "Ops Agent",
        "variant": "baseline",
        "dataset_version": "castalia_bench_v0.4",
        "quality_score": 0.72,
        "safety_score": 0.86,
        "human_score": 0.7,
        "latency_ms": 2100,
        "cost_usd": 0.041,
        "release_gate": "hold",
        "git_sha": "n7o8p9q",
        "seed": 21,
        "prompt_version": "agent_prompt@1",
        "eval_config": {"judge": "trajectory_qwen", "tools": 4},
        "case_scores": [0.65, 0.69, 0.76, 0.74, 0.76],
    },
    {
        "run_id": "agent_v2",
        "family": "agent",
        "system_name": "Ops Agent",
        "variant": "candidate",
        "dataset_version": "castalia_bench_v0.4",
        "quality_score": 0.79,
        "safety_score": 0.9,
        "human_score": 0.77,
        "latency_ms": 1880,
        "cost_usd": 0.039,
        "release_gate": "review",
        "git_sha": "r2s3t4u",
        "seed": 21,
        "prompt_version": "agent_prompt@2",
        "eval_config": {"judge": "trajectory_qwen", "tools": 4},
        "case_scores": [0.76, 0.8, 0.74, 0.81, 0.84],
    },
]

print("Runs loaded:", len(run_records))

## Step 2: Validate and normalize the schema

Before building reports, make sure every record has the fields needed for reproducibility and comparison.

In [ ]:
REQUIRED_FIELDS = {
    "run_id",
    "family",
    "system_name",
    "variant",
    "dataset_version",
    "quality_score",
    "safety_score",
    "human_score",
    "latency_ms",
    "cost_usd",
    "release_gate",
    "git_sha",
    "seed",
    "prompt_version",
    "eval_config",
    "case_scores",
}

def validate_run(record):
    missing = sorted(REQUIRED_FIELDS - set(record))
    if missing:
        raise ValueError(f'Missing required fields for {record.get("run_id", "unknown")}: {missing}')
    if not record["case_scores"]:
        raise ValueError(f'Run {record["run_id"]} has no case scores')
    return True

def config_hash(record):
    serialized = json.dumps(
        {
            "dataset_version": record["dataset_version"],
            "prompt_version": record["prompt_version"],
            "eval_config": record["eval_config"],
            "seed": record["seed"],
        },
        sort_keys=True,
    )
    return hashlib.md5(serialized.encode("utf-8")).hexdigest()[:10]

for record in run_records:
    validate_run(record)
    record["config_hash"] = config_hash(record)

print("All run records validated.")

## Step 3: Compute a composite score

Composite scores are not replacements for detailed analysis, but they help create a quick leaderboard view. We weight quality and safety most heavily, while still reflecting human review.

In [ ]:
def composite_score(record):
    return round(
        0.45 * record["quality_score"]
        + 0.35 * record["safety_score"]
        + 0.20 * record["human_score"],
        3,
    )

for record in run_records:
    record["composite_score"] = composite_score(record)

leaderboard_seed = [
    {
        "run_id": record["run_id"],
        "family": record["family"],
        "variant": record["variant"],
        "composite_score": record["composite_score"],
    }
    for record in run_records
]
print(to_markdown_table(leaderboard_seed, ["run_id", "family", "variant", "composite_score"]))

## Step 4: Build overall leaderboard tables

We sort the experiment family variants by composite score, then show operational metrics alongside quality.

In [ ]:
leaderboard_rows = []
for record in sorted(run_records, key=lambda row: row["composite_score"], reverse=True):
    leaderboard_rows.append(
        {
            "run_id": record["run_id"],
            "family": record["family"],
            "composite": record["composite_score"],
            "quality": record["quality_score"],
            "safety": record["safety_score"],
            "human": record["human_score"],
            "latency_ms": record["latency_ms"],
            "cost_usd": record["cost_usd"],
            "release_gate": record["release_gate"],
        }
    )

print(to_markdown_table(leaderboard_rows, ["run_id", "family", "composite", "quality", "safety", "human", "latency_ms", "cost_usd", "release_gate"]))

## Step 5: Compare candidates to their baselines

Release reviews need deltas, not just absolute scores.

In [ ]:
runs_by_family = defaultdict(dict)
for record in run_records:
    runs_by_family[record["family"]][record["variant"]] = record

comparison_rows = []
for family, variants in sorted(runs_by_family.items()):
    baseline = variants["baseline"]
    candidate = variants["candidate"]
    comparison_rows.append(
        {
            "family": family,
            "quality_delta": round(candidate["quality_score"] - baseline["quality_score"], 3),
            "safety_delta": round(candidate["safety_score"] - baseline["safety_score"], 3),
            "human_delta": round(candidate["human_score"] - baseline["human_score"], 3),
            "latency_delta_ms": candidate["latency_ms"] - baseline["latency_ms"],
            "cost_delta_usd": round(candidate["cost_usd"] - baseline["cost_usd"], 3),
            "gate_change": f'{baseline["release_gate"]} -> {candidate["release_gate"]}',
        }
    )

print(to_markdown_table(comparison_rows, ["family", "quality_delta", "safety_delta", "human_delta", "latency_delta_ms", "cost_delta_usd", "gate_change"]))

## Step 6: Add confidence intervals from case scores

Per-run case scores let us estimate uncertainty rather than over-trusting one average.

In [ ]:
def bootstrap_mean_interval(values, rounds=400):
    rng = random.Random(2100 + len(values))
    means = []
    for _ in range(rounds):
        sample = [rng.choice(values) for _ in values]
        means.append(statistics.fmean(sample))
    means.sort()
    lower = means[int(0.05 * rounds)]
    upper = means[int(0.95 * rounds) - 1]
    return round(lower, 3), round(upper, 3)

interval_rows = []
for record in run_records:
    lower, upper = bootstrap_mean_interval(record["case_scores"])
    record["ci_90"] = [lower, upper]
    interval_rows.append(
        {
            "run_id": record["run_id"],
            "mean": round(statistics.fmean(record["case_scores"]), 3),
            "ci_90": f"[{lower}, {upper}]",
        }
    )

print(to_markdown_table(interval_rows, ["run_id", "mean", "ci_90"]))

## Step 7: Build a reproducibility log

If a run cannot be reproduced, it is weak evidence. We gather the minimum metadata needed to rerun it later.

In [ ]:
timestamp = datetime.utcnow().isoformat(timespec="seconds") + "Z"

reproducibility_log = []
for record in run_records:
    reproducibility_log.append(
        {
            "run_id": record["run_id"],
            "git_sha": record["git_sha"],
            "dataset_version": record["dataset_version"],
            "prompt_version": record["prompt_version"],
            "seed": record["seed"],
            "config_hash": record["config_hash"],
            "logged_at": timestamp,
        }
    )

print(to_markdown_table(reproducibility_log, ["run_id", "git_sha", "dataset_version", "prompt_version", "seed", "config_hash", "logged_at"]))

## Step 8: Slice the leaderboard by operational constraints

Sometimes the highest-quality system is too slow or too expensive. Reporting should support filtered views, not only one global ranking.

In [ ]:
def constrained_leaderboard(records, max_latency_ms=None, max_cost_usd=None):
    filtered = []
    for record in records:
        if max_latency_ms is not None and record["latency_ms"] > max_latency_ms:
            continue
        if max_cost_usd is not None and record["cost_usd"] > max_cost_usd:
            continue
        filtered.append(record)
    return sorted(filtered, key=lambda row: row["composite_score"], reverse=True)

budget_rows = [
    {
        "run_id": record["run_id"],
        "family": record["family"],
        "composite": record["composite_score"],
        "latency_ms": record["latency_ms"],
        "cost_usd": record["cost_usd"],
    }
    for record in constrained_leaderboard(run_records, max_latency_ms=1500, max_cost_usd=0.03)
]

print(to_markdown_table(budget_rows, ["run_id", "family", "composite", "latency_ms", "cost_usd"]))

## Step 9: Generate a promotion report for each experiment family

A promotion report should summarize improvements, tradeoffs, and gating status in a compact artifact.

In [ ]:
promotion_reports = []
for family, variants in sorted(runs_by_family.items()):
    baseline = variants["baseline"]
    candidate = variants["candidate"]
    promote = (
        candidate["composite_score"] > baseline["composite_score"]
        and candidate["release_gate"] in {"ship", "review"}
        and candidate["safety_score"] >= baseline["safety_score"]
    )
    promotion_reports.append(
        {
            "family": family,
            "baseline_run": baseline["run_id"],
            "candidate_run": candidate["run_id"],
            "recommended_action": "promote_candidate" if promote else "keep_baseline",
            "reason": (
                "Candidate improved composite score without a safety drop."
                if promote else
                "Baseline remains safer or more reliable under current gate."
            ),
        }
    )

print(to_markdown_table(promotion_reports, ["family", "baseline_run", "candidate_run", "recommended_action", "reason"]))

## Step 10: Write report artifacts to disk

We save the raw runs, leaderboard, reproducibility log, and promotion report so they can be used outside the notebook.

In [ ]:
(ARTIFACT_DIR / "run_records.json").write_text(json.dumps(run_records, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "reproducibility_log.json").write_text(json.dumps(reproducibility_log, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "promotion_reports.json").write_text(json.dumps(promotion_reports, indent=2), encoding="utf-8")

leaderboard_text = to_markdown_table(
    leaderboard_rows,
    ["run_id", "family", "composite", "quality", "safety", "human", "latency_ms", "cost_usd", "release_gate"],
)
(ARTIFACT_DIR / "leaderboard.md").write_text(leaderboard_text, encoding="utf-8")

print("Saved reporting artifacts to", ARTIFACT_DIR.resolve())

## Step 11: Produce an executive summary

This final step creates the kind of short narrative that might appear in a release-review document.

In [ ]:
top_run = max(run_records, key=lambda row: row["composite_score"])
holds = [record["run_id"] for record in run_records if record["release_gate"] == "hold"]

executive_summary = {
    "top_run": top_run["run_id"],
    "top_family": top_run["family"],
    "top_composite_score": top_run["composite_score"],
    "runs_requiring_hold": holds,
    "families_ready_for_promotion": [
        report["family"]
        for report in promotion_reports
        if report["recommended_action"] == "promote_candidate"
    ],
}

print(json.dumps(executive_summary, indent=2))

## Recap

You now have a local reporting layer for Castalia Bench:

- structured run metadata
- leaderboard tables
- delta reports versus baselines
- reproducibility logs
- promotion recommendations

This turns evaluation outputs into operational evidence for iteration and release decisions.

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** design an eval dataset for a new use case. Document what changes and why.

**Exercise 2 — Build:** implement a custom metric and compare it to the one in this notebook. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** run the evaluation on a different model and analyze the differences.

## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the evals/ directory

---

## 🧭 Navigation

⬅️ **Previous:** [20 Human Eval Workflows](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/evals/20_human_eval_workflows.ipynb) | ➡️ **Next:** [22 Castalia Bench Capstone](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/evals/22_castalia_bench_capstone.ipynb)